# 05 - Engenharia de Features: Embeddings Contextuais BERT (BERTimbau)

**Objetivo:** Capturar a inteligência semântica dos cardápios. Diferente de One-Hot Encoding (que vê apenas palavras isoladas), o BERT entende o contexto (ex: "Frango com Quiabo" vs "Frango à Passarinho").

**Vantagem:** Permite que o modelo de predição aprenda a "atratividade" das refeições, identificando padrões de consumo baseados na complexidade e tipo de prato.

In [1]:
import pandas as pd
import numpy as np
import os
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

# Configurações
BASE_FEATURES = '../data/base_features_final.csv'
BASE_CARDAPIO = '../data/cardapio_consolidado.csv'
SAIDA_BERT = '../data/embeddings_bert_cardapio.csv'

# Modelo BERTimbau (Base) - Treinado para Português
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"

## 1. Preparação dos Textos do Cardápio
Concatenando os componentes da refeição para criar uma 'âncora semântica' por dia.

In [2]:
# Carregamento
df_feat = pd.read_csv(BASE_FEATURES)
df_card = pd.read_csv(BASE_CARDAPIO)

df_feat['data'] = pd.to_datetime(df_feat['data'])
df_card['data'] = pd.to_datetime(df_card['data'])

# Seleção de colunas textuais relevantes
text_cols = ['proteina_principal', 'preparo_principal', 'proteina_vegetariana', 
             'guarnicao', 'acompanhamento', 'sobremesa', 'salada']
text_cols = [c for c in text_cols if c in df_card.columns]

# Criar texto combinado
df_card['texto_combinado'] = df_card[text_cols].fillna('').agg(' '.join, axis=1).str.strip()

# Merge para processar apenas os dias que estão na base de features (2023-2025)
df_final = pd.merge(df_feat[['data']], df_card[['data', 'texto_combinado']], on='data', how='inner')

print(f"📝 Texto preparado para {len(df_final)} dias (Período Completo).")
print("Exemplo de entrada para o BERT:", df_final['texto_combinado'].iloc[0])

📝 Texto preparado para 191 dias (Período Completo).
Exemplo de entrada para o BERT: BOVINA ASSADO VEGETAL


## 2. Extração de Embeddings (Deep Learning)
Utilizando a média do 'Last Hidden State' para capturar a essência vetorial de cada 128 tokens.

In [3]:
# Carregar Modelo e Tokenizer
print("⏳ Carregando BERTimbau...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)

# GPU acelera muito o processo se disponível
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"Dispositivo de processamento: {device}")

def get_bert_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    # Média do último estado oculto (técnica recomendada para representação de sentenças)
    embeddings = outputs.last_hidden_state.mean(dim=1).cpu().numpy()
    return embeddings[0]

# Extração
embeddings_list = []
print(f"🚀 Extraindo embeddings para {len(df_final)} registros...")
for text in tqdm(df_final['texto_combinado']):
    embeddings_list.append(get_bert_embedding(text))

# Criar DataFrame de Embeddings
bert_columns = [f'bert_{i}' for i in range(len(embeddings_list[0]))]
df_bert = pd.DataFrame(embeddings_list, columns=bert_columns)
df_bert['data'] = df_final['data'].values

# Salvar
df_bert.to_csv(SAIDA_BERT, index=False)
print(f"✨ Embeddings BERT salvos em {SAIDA_BERT}")
print(f"📊 Dimensão: {df_bert.shape}")

⏳ Carregando BERTimbau...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Dispositivo de processamento: cpu
🚀 Extraindo embeddings para 191 registros...


100%|██████████| 191/191 [00:07<00:00, 24.12it/s]


✨ Embeddings BERT salvos em ../data/embeddings_bert_cardapio.csv
📊 Dimensão: (191, 769)
